# 10 — Statistical Significance Analysis

Rigorous evaluation of HPA, DQN, and PPO over **30 independent seeds**
(30 distinct random workloads). Because all three agents are evaluated on the
**same** seeds, the comparison is *paired*, enabling paired t-tests.

Produces:
- mean ± std per agent (the canonical report numbers)
- paired t-tests with p-values (statistical significance)
- percentage improvements with confidence

Uses the refactored modules: `env.py`, `agent.py`, `evaluate.py`.

## 1. Imports and load trained models

In [ ]:
import json
import numpy as np
import torch
from scipy import stats as scipy_stats

from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic, DQN
from evaluate import run_hpa, run_ppo, run_dqn

stats = json.load(open('trace_params.json'))['stats']

# load the freshly-trained canonical models
ppo_net = ActorCritic()
ppo_net.load_state_dict(torch.load('ppo_sla-focused.pth'))
ppo_net.eval()

dqn_net = DQN()
dqn_net.load_state_dict(torch.load('dqn_baseline.pth'))
dqn_net.eval()

print("Models loaded. Ready for 30-seed evaluation.")

## 2. Run all three agents over 30 paired seeds

Each agent is evaluated on the identical 30 workloads. Evaluation is fast
(no training) — expect ~1-2 minutes total.

In [ ]:
import time

N_SEEDS = 30
results = {'HPA': [], 'DQN': [], 'PPO': []}
t0 = time.time()

print(f"Running {N_SEEDS} paired seeds per agent...\n")
for i in range(N_SEEDS):
    seed = 1000 + i
    results['HPA'].append(run_hpa(CloudClusterEnv(stats, seed=seed)))
    results['DQN'].append(run_dqn(CloudClusterEnv(stats, seed=seed), dqn_net))
    results['PPO'].append(run_ppo(CloudClusterEnv(stats, seed=seed), ppo_net))
    if (i+1) % 5 == 0:
        print(f"  completed {i+1}/{N_SEEDS} seeds  ({time.time()-t0:.0f}s)")

json.dump(results, open('significance_results.json', 'w'), indent=2)
print(f"\nDone in {time.time()-t0:.0f}s. Saved significance_results.json")

## 3. Canonical numbers — mean ± std per agent

In [ ]:
print(f"{'agent':<6}{'cost (mean±std)':>22}{'breaches (mean±std)':>24}{'util':>10}")
print("-"*62)
for agent in ['HPA', 'DQN', 'PPO']:
    c = [r['cost'] for r in results[agent]]
    b = [r['breaches'] for r in results[agent]]
    u = [r['util'] for r in results[agent]]
    print(f"{agent:<6}{np.mean(c):>10.1f} ± {np.std(c):<8.1f}"
          f"{np.mean(b):>12.0f} ± {np.std(b):<8.0f}{np.mean(u):>10.3f}")

## 4. Statistical significance — paired t-tests

In [ ]:
def compare(a_name, b_name, metric):
    a = np.array([r[metric] for r in results[a_name]])
    b = np.array([r[metric] for r in results[b_name]])
    t_stat, p_value = scipy_stats.ttest_rel(a, b)   # paired
    return a.mean(), b.mean(), t_stat, p_value

print("="*70)
print("PAIRED t-TESTS (n=30 seeds)")
print("="*70)
for metric in ['cost', 'breaches']:
    print(f"\n--- {metric.upper()} ---")
    for a, b in [('PPO','HPA'), ('PPO','DQN'), ('DQN','HPA')]:
        ma, mb, t, p = compare(a, b, metric)
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
        better = a if ma < mb else b       # lower is better for both
        print(f"  {a} ({ma:.0f}) vs {b} ({mb:.0f}): t={t:.2f}, p={p:.2e} {sig}  -> {better} better")

print("\n" + "="*70)
print("*** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant")

## 5. PPO improvement over HPA — with confidence

In [ ]:
print("PPO improvement over HPA (per-seed paired, n=30):\n")
for metric in ['cost', 'breaches']:
    ppo_vals = np.array([r[metric] for r in results['PPO']])
    hpa_vals = np.array([r[metric] for r in results['HPA']])
    pct = (hpa_vals - ppo_vals) / hpa_vals * 100
    print(f"  {metric:<10}: {pct.mean():>5.1f}% ± {pct.std():.1f}% reduction")

# save a compact summary for the report / figures
summary = {}
for agent in ['HPA','DQN','PPO']:
    summary[agent] = {
        m: {'mean': float(np.mean([r[m] for r in results[agent]])),
            'std':  float(np.std([r[m] for r in results[agent]]))}
        for m in ['cost','breaches','util','vms']
    }
json.dump(summary, open('significance_summary.json', 'w'), indent=2)
print("\nSaved significance_summary.json")

## 6. Stronger baseline — proportional (Kubernetes-style) HPA

The threshold baseline above steps ±1 VM on fixed CPU thresholds. Real
Kubernetes HPA is proportional: `desired = ceil(current × util / target)`,
with a 10% tolerance band. This section evaluates that algorithm on the same
30 paired seeds, at three target utilisations (giving the baseline its best
configuration), against the already-saved PPO results.

Result summary: PPO remains far cheaper (~23–31%), but proportional HPA at
conservative targets achieves far fewer breaches — neither dominates; they
occupy different points on the cost–SLA frontier. Note the simulator has no
VM provisioning delay, which favours reactive controllers — proportional
HPA's breach numbers here are an upper bound on its real-world performance.

(The cell below is self-contained — it can be run alone in a fresh kernel.)

In [1]:
# Proportional HPA over the same 30 paired seeds, three target utilisations.
# Fully self-contained: safe to run alone in a fresh kernel.
import json
import numpy as np
from scipy import stats as scipy_stats
from env import CloudClusterEnv
from evaluate import run_hpa_proportional

stats = json.load(open('trace_params.json'))['stats']
saved = json.load(open('significance_results.json'))
ppo_c = np.array([r['cost'] for r in saved['PPO']])
ppo_b = np.array([r['breaches'] for r in saved['PPO']])

prop_results = {}
for target in [0.5, 0.6, 0.7]:
    runs = [run_hpa_proportional(CloudClusterEnv(stats, seed=1000+i), target=target)
            for i in range(30)]
    prop_results[str(target)] = runs

json.dump(prop_results, open('proportional_hpa_results.json', 'w'), indent=2)
print("Saved proportional_hpa_results.json\n")

print(f"{'agent':<22}{'cost (mean±std)':>20}{'breaches (mean±std)':>24}{'vms':>8}")
print("-"*74)
thr_c = [r['cost'] for r in saved['HPA']]; thr_b = [r['breaches'] for r in saved['HPA']]
print(f"{'threshold HPA':<22}{np.mean(thr_c):>12.1f} ± {np.std(thr_c):<5.1f}"
      f"{np.mean(thr_b):>14.0f} ± {np.std(thr_b):<7.0f}")
for target, runs in prop_results.items():
    c = [r['cost'] for r in runs]; b = [r['breaches'] for r in runs]
    print(f"{'proportional t='+target:<22}{np.mean(c):>12.1f} ± {np.std(c):<5.1f}"
          f"{np.mean(b):>14.0f} ± {np.std(b):<7.0f}{np.mean([r['vms'] for r in runs]):>8.1f}")
print(f"{'PPO (sla-focused)':<22}{ppo_c.mean():>12.1f} ± {ppo_c.std():<5.1f}"
      f"{ppo_b.mean():>14.0f} ± {ppo_b.std():<7.0f}")

print("\nPaired t-tests, PPO vs proportional HPA (n=30):")
for target, runs in prop_results.items():
    pc = np.array([r['cost'] for r in runs]); pb = np.array([r['breaches'] for r in runs])
    tc, pvc = scipy_stats.ttest_rel(ppo_c, pc)
    tb, pvb = scipy_stats.ttest_rel(ppo_b, pb)
    cost_pct = (pc.mean() - ppo_c.mean()) / pc.mean() * 100
    print(f"  t={target}: PPO cost {cost_pct:.1f}% lower (p={pvc:.1e}); "
          f"proportional HPA breaches lower (p={pvb:.1e})")

print("\nInterpretation: PPO Pareto-dominates the threshold baseline "
      "(cheaper AND fewer breaches). Against proportional HPA neither "
      "dominates: PPO is ~23-31% cheaper, proportional HPA breaches less — "
      "they occupy different points on the cost-SLA frontier. Caveat: the "
      "simulator provisions VMs instantly, which favours reactive scaling; "
      "with realistic spin-up delay proportional HPA's breach numbers "
      "would be worse (an upper bound on its real performance).")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Saved proportional_hpa_results.json

agent                      cost (mean±std)     breaches (mean±std)     vms
--------------------------------------------------------------------------
threshold HPA                291.1 ± 2.3            1785 ± 205    
proportional t=0.5           288.0 ± 1.1               0 ± 0           8.6
proportional t=0.6           256.3 ± 1.0               4 ± 18          7.6
proportional t=0.7           231.7 ± 0.7             183 ± 126         6.9
PPO (sla-focused)            177.7 ± 0.7             878 ± 312    

Paired t-tests, PPO vs proportional HPA (n=30):
  t=0.5: PPO cost 38.3% lower (p=6.3e-60); proportional HPA breaches lower (p=2.5e-15)
  t=0.6: PPO cost 30.6% lower (p=6.7e-56); proportional HPA breaches lower (p=1.1e-15)
  t=0.7: PPO cost 23.3% lower (p=1.4e-56); proportional HPA breaches lower (p=5.5e-16)

Interpretation: PPO Pareto-dominates the threshold baseline (cheaper AND fewer bre

## 7. Robustness check — strict enforcement of the ±5 VMs/step limit

The actor's deterministic mean occasionally lies outside the nominal [-1,1]
action box (~17% of steps — almost all deep scale-down commands issued while
already at the 2-VM floor, where they have no effect). Since `env.step()`
does not clip the action, a single step can in principle change VMs by more
than 5. This check re-evaluates the canonical policy with the action clipped
to the box, strictly enforcing ±5 VMs/step, on the same 30 paired seeds.

Result: cost and breaches shift by ~1%, confirming the PPO advantage does
not depend on exceeding the nominal step limit. (Self-contained cell.)

In [1]:
# Robustness check: strict ±5/step via action clipping. Self-contained.
import json
import numpy as np
import torch
from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic

stats = json.load(open('trace_params.json'))['stats']
net = ActorCritic()
net.load_state_dict(torch.load('ppo_sla-focused.pth'))
net.eval()

def run_clipped(seed):
    env = CloudClusterEnv(stats, seed=seed)
    obs, _ = env.reset()
    obs = torch.tensor(obs, dtype=torch.float32)
    cost = breaches = 0.0
    for t in range(STEPS_PER_WEEK):
        with torch.no_grad():
            mean, _ = net.forward(obs.unsqueeze(0))
        a = np.clip(mean.squeeze(0).numpy(), -1.0, 1.0)   # enforce the box
        obs, r, done, tr, info = env.step(a)
        obs = torch.tensor(obs, dtype=torch.float32)
        cost += info['cost']; breaches += info['breaches']
    return cost, breaches

rows = [run_clipped(1000 + i) for i in range(30)]
cc = np.array([r[0] for r in rows]); cb = np.array([r[1] for r in rows])

saved = json.load(open('significance_results.json'))
uc = np.array([r['cost'] for r in saved['PPO']])
ub = np.array([r['breaches'] for r in saved['PPO']])

print(f"unclipped (canonical): cost {uc.mean():.1f} ± {uc.std():.1f}   "
      f"breaches {ub.mean():.0f} ± {ub.std():.0f}")
print(f"clipped to ±5/step:    cost {cc.mean():.1f} ± {cc.std():.1f}   "
      f"breaches {cb.mean():.0f} ± {cb.std():.0f}")
print(f"delta: cost {(cc.mean()-uc.mean())/uc.mean()*100:+.1f}%, "
      f"breaches {(cb.mean()-ub.mean())/ub.mean()*100:+.1f}%")

json.dump({'clipped': [{'cost': float(c), 'breaches': float(b)} for c, b in rows]},
          open('clipped_action_check.json', 'w'), indent=2)
print("\nSaved clipped_action_check.json. Conclusion: the PPO advantage does "
      "not depend on exceeding the nominal ±5 VMs/step limit.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
unclipped (canonical): cost 177.7 ± 0.7   breaches 878 ± 312
clipped to ±5/step:    cost 178.8 ± 0.8   breaches 893 ± 344
delta: cost +0.6%, breaches +1.7%

Saved clipped_action_check.json. Conclusion: the PPO advantage does not depend on exceeding the nominal ±5 VMs/step limit.


In [2]:
import time, os, json
import numpy as np
import torch
from env import CloudClusterEnv
from agent import ActorCritic, DQN

stats = json.load(open('trace_params.json'))['stats']

ppo_net = ActorCritic(); ppo_net.load_state_dict(torch.load('ppo_sla-focused.pth')); ppo_net.eval()
dqn_net = DQN(); dqn_net.load_state_dict(torch.load('dqn_baseline.pth')); dqn_net.eval()

env = CloudClusterEnv(stats, seed=1)
obs, _ = env.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

# ---- inference latency ----
def hpa_decision(cpu):
    return 0.2 if cpu > 0.70 else (-0.2 if cpu < 0.30 else 0.0)

n = 1000
# warmup
for _ in range(10):
    with torch.no_grad(): ppo_net.forward(obs_t.unsqueeze(0))

t0 = time.perf_counter()
for _ in range(n): hpa_decision(obs_t[0].item())
hpa_lat = (time.perf_counter()-t0)/n*1000

t0 = time.perf_counter()
for _ in range(n):
    with torch.no_grad(): ppo_net.forward(obs_t.unsqueeze(0))
ppo_lat = (time.perf_counter()-t0)/n*1000

t0 = time.perf_counter()
for _ in range(n):
    with torch.no_grad(): dqn_net(obs_t.unsqueeze(0))
dqn_lat = (time.perf_counter()-t0)/n*1000

# ---- resource overhead ----
ppo_params = sum(p.numel() for p in ppo_net.parameters())
dqn_params = sum(p.numel() for p in dqn_net.parameters())
ppo_size = os.path.getsize('ppo_sla-focused.pth')/1024
dqn_size = os.path.getsize('dqn_baseline.pth')/1024

# Note on scalability: the state is a fixed 32-dim summary (histories, queue
# ratios, temporal features), independent of how many VMs/jobs the cluster
# has — so inference cost is O(1) with respect to cluster size by
# construction. This is an architectural property, not something to benchmark.

metrics = {
    'latency_ms': {'HPA': hpa_lat, 'DQN': dqn_lat, 'PPO': ppo_lat},
    'params': {'HPA': 0, 'DQN': dqn_params, 'PPO': ppo_params},
    'model_kb': {'HPA': 0, 'DQN': dqn_size, 'PPO': ppo_size},
}
json.dump(metrics, open('system_metrics.json', 'w'), indent=2)
print("Saved system_metrics.json")
print(f"Latency (ms): HPA {hpa_lat:.4f}, DQN {dqn_lat:.4f}, PPO {ppo_lat:.4f}")
print(f"Params: DQN {dqn_params:,}, PPO {ppo_params:,}")

Saved system_metrics.json
Latency (ms): HPA 0.0011, DQN 0.0192, PPO 0.0290
Params: DQN 75,525, PPO 74,755
